In [1]:
import numpy as np
import urllib.request
import zipfile
import os
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Flatten

In [2]:
# 1. 데이터 로드 (2번 링크 과정)
vocab_size = 10000
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

max_len = 500
X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

# 단어 사전 준비
word_index = imdb.get_word_index()
# IMDB 인덱스는 3씩 밀려있으므로 조정이 필요합니다.
index_to_word = {v+3: k for k, v in word_index.items()}
index_to_word[0] = "<pad>"
index_to_word[1] = "<sos>"
index_to_word[2] = "<unk>"
actual_word_index = {k: v for v, k in index_to_word.items()}

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step


In [3]:
# 2. GloVe 다운로드 및 로드 (1번 링크 과정)
urllib.request.urlretrieve("http://nlp.stanford.edu/data/glove.6B.zip", filename="glove.6B.zip")
with zipfile.ZipFile("glove.6B.zip", 'r') as zip_ref:
    zip_ref.extractall()

embedding_dict = dict()
with open('glove.6B.100d.txt', encoding="utf8") as f:
    for line in f:
        word_vector = line.split()
        word = word_vector[0]
        word_vector_arr = np.asarray(word_vector[1:], dtype='float32')
        embedding_dict[word] = word_vector_arr

In [4]:
# 3. 임베딩 행렬 생성 (힌트 링크 내용 참고)
embedding_matrix = np.zeros((vocab_size, 100))

for word, index in actual_word_index.items():
    if index < vocab_size:
        temp = embedding_dict.get(word)
        if temp is not None:
            embedding_matrix[index] = temp

In [5]:
# 4. 모델 설계 및 훈련 (GRU 제외 버전)
model = Sequential()
model.add(Embedding(vocab_size, 100, weights=[embedding_matrix], input_length=max_len, trainable=False))
model.add(Flatten()) # GRU 대신 사용
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['acc'])
model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.2)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - acc: 0.6653 - loss: 0.6473 - val_acc: 0.7050 - val_loss: 0.5969
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - acc: 0.8348 - loss: 0.3716 - val_acc: 0.7308 - val_loss: 0.5766
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - acc: 0.8814 - loss: 0.2886 - val_acc: 0.7216 - val_loss: 0.6166
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - acc: 0.9147 - loss: 0.2324 - val_acc: 0.7178 - val_loss: 0.6584
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - acc: 0.9298 - loss: 0.2024 - val_acc: 0.6866 - val_loss: 0.8160
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - acc: 0.9438 - loss: 0.1720 - val_acc: 0.7256 - val_loss: 0.7115
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - acc: 0.9592 - loss: 0.1459 - val_acc: 0.7148 - val_loss: 0.7631
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - acc: 0.9651 - loss: 0.1309 - val_acc: 0.7194 - val_loss: 0.7781
Epoch 9/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step